# Kepler's Third Law — Orbital Period vs. Distance from the Sun

Kepler's Third Law states that the square of a planet's orbital period is proportional to the cube of its average distance from the Sun:

$$T^2 \propto a^3$$

When plotted on a **log-log scale**, this becomes a straight line with slope **3/2**. Let's visualize it with real planetary (and dwarf planet) data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np  # only for linspace/log — no heavy deps

In [ ]:
# ── Data ─────────────────────────────────────────────────────────────────────
# Semi-major axis in AU, orbital period in Earth years
data = {
    "body":   ["Mercury", "Venus",  "Earth",  "Mars",   "Ceres",
               "Jupiter", "Saturn", "Uranus", "Neptune", "Pluto"],
    "au":     [0.387,     0.723,    1.000,    1.524,    2.767,
               5.203,     9.537,    19.191,   30.069,   39.482],
    "period": [0.241,     0.615,    1.000,    1.881,    4.599,
               11.862,    29.457,   84.011,   164.795,  247.94],
    "type":   ["Planet",  "Planet", "Planet", "Planet", "Dwarf",
               "Planet",  "Planet", "Planet", "Planet", "Dwarf"],
}

df = pd.DataFrame(data)
df.head(10)

In [ ]:
# ── Kepler's predicted curve  (T = a^1.5) ────────────────────────────────────
a_range = np.linspace(0.3, 42, 300)
t_kepler = a_range ** 1.5

# ── Colour & marker maps ──────────────────────────────────────────────────────
COLORS  = {"Planet": "#4C9BE8", "Dwarf": "#E87B4C"}
MARKERS = {"Planet": "o",       "Dwarf": "D"}

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor("#0D1117")
ax.set_facecolor("#0D1117")

# Kepler theory line
ax.plot(a_range, t_kepler,
        color="#FFD700", linewidth=1.8, linestyle="--",
        label=r"Kepler: $T = a^{3/2}$", zorder=1)

# Scatter per body-type
for btype, group in df.groupby("type"):
    ax.scatter(group["au"], group["period"],
               color=COLORS[btype], marker=MARKERS[btype],
               s=90, zorder=3, label=btype, edgecolors="white", linewidths=0.5)

# Labels for each body
offsets = {
    "Mercury": (-0.01, -4),  "Venus":   (0.05, -7),
    "Earth":   (0.1,   -8),  "Mars":    (0.12, -5),
    "Ceres":   (0.2,   -4),  "Jupiter": (0.4,  -9),
    "Saturn":  (0.6,  -12),  "Uranus":  (1.2, -18),
    "Neptune": (1.5,  -22),  "Pluto":   (-8,   10),
}
for _, row in df.iterrows():
    dx, dy = offsets.get(row["body"], (0.2, 0))
    ax.annotate(row["body"],
                xy=(row["au"], row["period"]),
                xytext=(row["au"] + dx, row["period"] + dy),
                color="white", fontsize=8.5,
                arrowprops=dict(arrowstyle="-", color="#888888", lw=0.7))

# Axes formatting
ax.set_xlabel("Semi-major Axis (AU)", color="white", fontsize=12)
ax.set_ylabel("Orbital Period (Earth years)", color="white", fontsize=12)
ax.set_title("Kepler's Third Law  —  Solar System Bodies",
             color="white", fontsize=15, fontweight="bold", pad=14)

ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444444")

ax.grid(True, color="#2A2A2A", linestyle=":", linewidth=0.8)

legend = ax.legend(facecolor="#1A1F2E", edgecolor="#444444",
                   labelcolor="white", fontsize=10)

plt.tight_layout()
plt.savefig("kepler_third_law.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved → kepler_third_law.png")

## Verification — log-log linearization

On a log-log plot Kepler's law is a straight line with slope **1.5**.  
Let's confirm with a quick check on the actual data.

In [ ]:
df["log_au"]     = np.log10(df["au"])
df["log_period"] = np.log10(df["period"])

# Simple slope estimate from first/last point
slope = (df["log_period"].iloc[-1] - df["log_period"].iloc[0]) / \
        (df["log_au"].iloc[-1]     - df["log_au"].iloc[0])

fig2, ax2 = plt.subplots(figsize=(8, 5))
fig2.patch.set_facecolor("#0D1117")
ax2.set_facecolor("#0D1117")

ax2.scatter(df["log_au"], df["log_period"],
            color="#4C9BE8", s=80, zorder=3, edgecolors="white", linewidths=0.5)

x_fit = np.linspace(df["log_au"].min() - 0.1, df["log_au"].max() + 0.1, 100)
ax2.plot(x_fit, 1.5 * x_fit, color="#FFD700", linewidth=1.8,
         linestyle="--", label=f"slope = 1.5 (theory)")

ax2.set_xlabel(r"$\log_{10}$(AU)",       color="white", fontsize=12)
ax2.set_ylabel(r"$\log_{10}$(Period yr)", color="white", fontsize=12)
ax2.set_title("Log-Log Confirmation of Kepler's Third Law",
              color="white", fontsize=13, fontweight="bold")
ax2.tick_params(colors="white")
for spine in ax2.spines.values():
    spine.set_edgecolor("#444444")
ax2.grid(True, color="#2A2A2A", linestyle=":", linewidth=0.8)
ax2.legend(facecolor="#1A1F2E", edgecolor="#444444",
           labelcolor="white", fontsize=10)

plt.tight_layout()
plt.show()

print(f"Empirical slope from data endpoints: {slope:.4f}  (expected: 1.5)")